# 10 Loocv Validation

Leave-One-Out Cross-Validation for Exoplanet Classifier (4-class)
=====================================================================
Suggested notebook name: 10_loocv_validation.ipynb (place in notebooks/)

WHY: With a small labeled set, a single train/test split is meaningless.
LOOCV trains on every-sample-but-one, predicts the held-out one, repeats.
RF/XGB hyperparameters here match notebook 05 exactly (n_estimators=200,
XGB max_depth=6/learning_rate=0.05) so results are directly comparable.

Paths assume this runs from inside notebooks/ (matches every other
notebook in the project).

Confirmed against your actual notebooks:
    - feature_cols.pkl / label_encoder.pkl saved via plain pickle.dump
    - target column in labeled_features.csv is 'label'
    - tic_id needs the .0-strip fix on every CSV load

In [1]:
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import LeaveOneOut
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb

TARGET_COL = "label"
DATA_PATH = "../outputs/labeled_features.csv"
FEATURE_COLS_PATH = "../models/feature_cols.pkl"
OUTPUT_PATH = "../outputs/loocv_results.csv"

with open(FEATURE_COLS_PATH, "rb") as f:
    feature_cols = pickle.load(f)

df = pd.read_csv(DATA_PATH)
df["tic_id"] = df["tic_id"].astype(str).str.replace(".0", "", regex=False)

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing feature columns in {DATA_PATH}: {missing}\nAvailable: {list(df.columns)}")
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not in {DATA_PATH}. Available: {list(df.columns)}")

X = df[feature_cols].fillna(0).values
y_raw = df[TARGET_COL].values

le = LabelEncoder()
y = le.fit_transform(y_raw)

print(f"Classes: {list(le.classes_)}")
print(f"Samples per class: {dict(zip(le.classes_, np.bincount(y)))}")
print("NOTE: a class with very few samples will naturally score poorly in LOOCV --")
print("that's an honest reflection of too little data, not a bug in this script.\n")

loo = LeaveOneOut()


def run_loocv(model_fn, model_name):
    preds, trues = [], []
    for train_idx, test_idx in loo.split(X):
        model = model_fn()
        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[test_idx])
        preds.append(pred[0])
        trues.append(y[test_idx][0])

    preds, trues = np.array(preds), np.array(trues)
    acc = accuracy_score(trues, preds)

    print(f"\n{'=' * 55}")
    print(f"{model_name} -- Leave-One-Out Cross-Validation")
    print(f"{'=' * 55}")
    print(f"Accuracy: {acc:.3f}  ({(trues == preds).sum()}/{len(trues)} correct)")
    print("\nConfusion Matrix:")
    print(confusion_matrix(trues, preds, labels=range(len(le.classes_))))
    print("\nClassification Report:")
    print(classification_report(
        trues, preds, target_names=le.classes_,
        labels=range(len(le.classes_)), zero_division=0,
    ))

    return acc, preds, trues


# Hyperparameters match notebook 05 exactly for a fair comparison
rf_acc, rf_preds, rf_trues = run_loocv(
    lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(
            n_estimators=200, max_depth=None, min_samples_split=2,
            class_weight="balanced", random_state=42,
        )),
    ]),
    "Random Forest",
)

xgb_acc, xgb_preds, xgb_trues = run_loocv(
    lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("clf", xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.05,
            eval_metric="mlogloss", random_state=42,
        )),
    ]),
    "XGBoost",
)

results_df = pd.DataFrame({
    "tic_id": df["tic_id"].values,
    "true_label": le.inverse_transform(rf_trues),
    "rf_pred": le.inverse_transform(rf_preds),
    "xgb_pred": le.inverse_transform(xgb_preds),
})
results_df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved per-sample LOOCV predictions -> {OUTPUT_PATH}")

Classes: ['eclipsing_binary', 'planet']
Samples per class: {'eclipsing_binary': np.int64(5), 'planet': np.int64(10)}
NOTE: a class with very few samples will naturally score poorly in LOOCV --
that's an honest reflection of too little data, not a bug in this script.


Random Forest -- Leave-One-Out Cross-Validation
Accuracy: 0.800  (12/15 correct)

Confusion Matrix:
[[ 2  3]
 [ 0 10]]

Classification Report:
                  precision    recall  f1-score   support

eclipsing_binary       1.00      0.40      0.57         5
          planet       0.77      1.00      0.87        10

        accuracy                           0.80        15
       macro avg       0.88      0.70      0.72        15
    weighted avg       0.85      0.80      0.77        15


XGBoost -- Leave-One-Out Cross-Validation
Accuracy: 0.667  (10/15 correct)

Confusion Matrix:
[[1 4]
 [1 9]]

Classification Report:
                  precision    recall  f1-score   support

eclipsing_binary       0.50      0.20      0